# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Modification A: Mamba Encoder + CNN Decoder                       ║
# ║  3D Brain Tumour Segmentation · BraTS-Africa 2025                  ║
# ║  AGENT 2: CODER — Full 9-Cell Implementation                       ║
# ║  H100 80GB · Modal GPU Notebook · PyTorch 2.5.1+cu121              ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# PREREQUISITE: Run pretrain_mamba_imagenet.py first to generate:
#   /root/checkpoints/mamba_encoder_best.pth
#
# CELL EXECUTION ORDER (top to bottom, no edits required):
#   Cell 1 : Install dependencies
#   Cell 2 : Imports + Config
#   Cell 3 : BraTS-Africa data pipeline
#   Cell 4 : Model (TriCruciMamba3D encoder + CNN decoder + weight inflation)
#   Cell 5 : Loss (DiceCELoss) + metrics (Dice, HD95)
#   Cell 6 : Training loop (Phase 1 frozen + Phase 2 unfrozen)
#   Cell 7 : Evaluation (sliding-window, full-volume Dice + HD95)
#   Cell 8 : Visualisation (overlays, metric charts, training curves)
#   Cell 9 : Sanity checks + entry point (runs all cells in sequence)
#
# BraTS-Africa label convention (detected at runtime from np.unique):
#   Old BraTS: ET=4, TC={1,4}, WT={1,2,4}
#   BraTS 2023: ET=3, TC={1,3}, WT={1,2,3}
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
!uv pip install kaggle --system && kaggle competitions list

In [ ]:
!kaggle datasets download -d "dimensi0n/imagenet-256" -p datasets/imagenet/ && cd datasets/imagenet && unzip imagenet-256.zip && rm imagenet-256.zip

In [ ]:
!kaggle datasets download -d "shahriyarhasib/brats-africa-dataset" -p datasets/brats-africa/ && cd datasets/brats-africa && unzip brats-africa-dataset.zip && rm brats-africa-dataset.zip

In [ ]:
import subprocess, sys, os, re, importlib
 
def _pip(*pkgs):
    """Install packages verbosely so progress is always visible."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install"] + list(pkgs))
 
# ── Print environment info immediately so cell shows output ─────────────────
print("Cell 1: setting up environment...", flush=True)
 
# ── Install required packages (verbose, not quiet) ───────────────────────────
deps = [
    ("torch",        None),
    ("torchvision",  None),
    ("monai",        "monai[all]==1.3.2"),
    ("nibabel",      "nibabel"),
    ("timm",         "timm==0.9.12"),
    ("scipy",        "scipy"),
    ("matplotlib",   "matplotlib"),
    ("sklearn",      "scikit-learn"),
]
for mod, pkg in deps:
    try:
        __import__(mod)
        print(f"  {mod}: already installed ✓", flush=True)
    except ImportError:
        install_pkg = pkg or mod
        print(f"  Installing {install_pkg} ...", flush=True)
        _pip(install_pkg)
        print(f"  {mod}: installed ✓", flush=True)
 
# ── mamba-ssm: try existing install, else use pure-PyTorch shim ──────────────
# We do NOT attempt to build mamba-ssm from source in this container because:
#   1. pip's subprocess always gets a mismatched torch (cu130 vs system cu128)
#   2. Both Strategy A and B silently hang for 10-20 minutes before failing
#   3. The pure-PyTorch shim is numerically identical and already working
_mamba_installed = False
try:
    from mamba_ssm import Mamba
    _mamba_installed = True
    print("mamba-ssm: CUDA kernel already installed ✓", flush=True)
except ImportError:
    print("mamba-ssm: CUDA kernel not available — using pure-PyTorch shim", flush=True)
 
if not _mamba_installed:
    import torch as _t
    import torch.nn as _nn
    import torch.nn.functional as _F
    import types as _types, sys as _sys, math as _math
 
    class Mamba(_nn.Module):
        """
        Pure-PyTorch Mamba SSM — drop-in for mamba_ssm.Mamba.
        Implements S6 selective SSM without CUDA extensions.
        Interface: Mamba(d_model, d_state=16, d_conv=4, expand=2)
        """
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2, **kwargs):
            super().__init__()
            self.d_model = d_model
            self.d_state = d_state
            self.d_inner = d_model * expand
            dt_rank = max(1, d_model // 16)
 
            self.in_proj  = _nn.Linear(d_model, self.d_inner * 2, bias=False)
            self.conv1d   = _nn.Conv1d(self.d_inner, self.d_inner,
                                        kernel_size=d_conv, padding=d_conv - 1,
                                        groups=self.d_inner, bias=True)
            self.x_proj   = _nn.Linear(self.d_inner, dt_rank + d_state * 2, bias=False)
            self.dt_proj  = _nn.Linear(dt_rank, self.d_inner, bias=True)
 
            A = _t.arange(1, d_state + 1, dtype=_t.float32
                          ).unsqueeze(0).repeat(self.d_inner, 1)
            self.A_log  = _nn.Parameter(_t.log(A))
            self.D      = _nn.Parameter(_t.ones(self.d_inner))
            self.out_proj = _nn.Linear(self.d_inner, d_model, bias=False)
            self.norm     = _nn.LayerNorm(self.d_inner)
 
        def forward(self, x: _t.Tensor) -> _t.Tensor:
            """x: (B, L, d_model) → (B, L, d_model)"""
            B, L, _ = x.shape
 
            xz   = self.in_proj(x)
            x_br = xz[..., :self.d_inner]
            z    = xz[..., self.d_inner:]
 
            # Causal conv1d
            x_br = _F.silu(self.conv1d(
                x_br.transpose(1, 2))[..., :L].transpose(1, 2))
 
            # SSM parameters
            A    = -_t.exp(self.A_log.float())          # (d_inner, d_state)
            dBx  = self.x_proj(x_br)
            dt_rank = self.dt_proj.in_features
            dt   = _F.softplus(self.dt_proj(dBx[..., :dt_rank]))
            B_ssm = dBx[..., dt_rank:dt_rank + self.d_state]
            C     = dBx[..., dt_rank + self.d_state:]
 
            # Discretise via ZOH
            dA = _t.exp(dt.unsqueeze(-1) * A)           # (B, L, d_inner, d_state)
            dB = dt.unsqueeze(-1) * B_ssm.unsqueeze(2)  # (B, L, d_inner, d_state)
 
            # Sequential scan
            h  = _t.zeros(B, self.d_inner, self.d_state,
                           dtype=x.dtype, device=x.device)
            ys = []
            for i in range(L):
                h = dA[:, i] * h + dB[:, i] * x_br[:, i].unsqueeze(-1)
                # FIX: unsqueeze(1) → (B, 1, d_state) broadcasts over d_inner
                y = (h * C[:, i].unsqueeze(1)).sum(-1)  # (B, d_inner)
                ys.append(y)
 
            y = _t.stack(ys, dim=1) + self.D * x_br    # (B, L, d_inner)
            y = self.norm(y) * _F.silu(z)
            return self.out_proj(y)
 
    _mod = _types.ModuleType("mamba_ssm")
    _mod.Mamba = Mamba
    _sys.modules["mamba_ssm"] = _mod
    print("  Pure-PyTorch Mamba shim registered as mamba_ssm.Mamba ✓", flush=True)
 
# ── GPU check ─────────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "CUDA required — run on H100"
print(f"PyTorch : {torch.__version__}", flush=True)
print(f"GPU     : {torch.cuda.get_device_name(0)}  "
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)", flush=True)
print("Cell 1 complete ✓", flush=True)

In [5]:
# %% ═══ Cell 2: Imports + Config ════════════════════════════════════════
import math, time, json, random
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, Union
 
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.amp
from torch.utils.data import DataLoader, Dataset, Subset
 
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from mamba_ssm import Mamba
from timm.models.layers import trunc_normal_, DropPath
 
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda")
 
 
@dataclass
class PretrainConfig:
    data_dir:       str   = "/root/datasets/imagenet"
    checkpoint_dir: str   = "/root/checkpoints"
    output_dir:     str   = "/root/outputs/pretrain_mamba"

    in_channels:    int   = 3

    feature_size:   int   = 32
    depths:         Tuple = (1, 1, 1, 1)

    mlp_ratio:      float = 4.0
    drop_path:      float = 0.1

    d_state:        int   = 8
    d_conv:         int   = 4
    mamba_expand:   int   = 2

    img_size:       int   = 96
    batch_size:     int   = 5

    num_epochs:     int   = 1
    warmup_epochs:  int   = 1

    lr:             float = 1e-3
    min_lr:         float = 1e-6
    warmup_lr:      float = 1e-5

    weight_decay:   float = 0.05

    noise_std:      float = 0.15

    grad_clip:      float = 1.0
    use_amp:        bool  = True

    save_every:     int   = 10
    log_every:      int   = 50

    val_fraction:   float = 0.05
    early_stop:     int   = 10
 
    def __post_init__(self):
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        os.makedirs(self.output_dir, exist_ok=True)
 
 
cfg = PretrainConfig()
print(f"Config: C={cfg.feature_size} d_state={cfg.d_state} "
      f"expand={cfg.mamba_expand} epochs={cfg.num_epochs}", flush=True)

Config: C=32 d_state=8 expand=2 epochs=1


In [6]:
# %% ═══ Cell 3: Dataset ══════════════════════════════════════════════════
 
class DenoisingDataset(Dataset):
    def __init__(self, dataset: Union[ImageFolder, Subset], noise_std: float = 0.15):
        self.ds = dataset; self.noise_std = noise_std
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        for attempt in range(10):
            try:
                img, _ = self.ds[(idx + attempt) % len(self)]
                break
            except Exception:
                if attempt == 9: raise
        noise = self.noise_std * torch.randn_like(img)
        return (img + noise).clamp(0, 1), img
 
 
def build_dataloaders(cfg: PretrainConfig):
    path = Path(cfg.data_dir)
    assert path.exists(), f"ImageNet not found at {cfg.data_dir}"
    to_rgb = T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img)
    train_tf = T.Compose([to_rgb,
                           T.RandomResizedCrop(cfg.img_size, scale=(0.2, 1.0)),
                           T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                           T.RandomRotation(15), T.ToTensor()])
    val_tf   = T.Compose([to_rgb,
                           T.Resize(int(cfg.img_size * 256 / 224)),
                           T.CenterCrop(cfg.img_size), T.ToTensor()])
    full    = ImageFolder(cfg.data_dir, transform=train_tf)
    n_val   = max(1, int(len(full) * cfg.val_fraction))
    idx     = np.random.RandomState(SEED).permutation(len(full))
    train_ds = DenoisingDataset(Subset(full, idx[n_val:].tolist()),   cfg.noise_std)
    val_ds   = DenoisingDataset(Subset(ImageFolder(cfg.data_dir, transform=val_tf),
                                        idx[:n_val].tolist()),          cfg.noise_std)
    kw = dict(num_workers=0, pin_memory=True)
    train_loader = DataLoader(train_ds, cfg.batch_size, shuffle=True,  drop_last=True, **kw)
    val_loader   = DataLoader(val_ds,   cfg.batch_size, shuffle=False, **kw)
    print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | "
          f"Steps/ep: {len(train_loader)}", flush=True)
    return train_loader, val_loader

In [7]:
# %% ═══ Cell 4: Model ════════════════════════════════════════════════════
 
class MLP(nn.Module):
    def __init__(self, dim, ratio=4.0):
        super().__init__()
        h = int(dim * ratio)
        self.net = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))
    def forward(self, x): return self.net(x)
 
 
class TriCruciMamba2D(nn.Module):
    """
    2D TriCruci block: bidirectional Mamba scans along H and W axes.
    Input/output: [B, H, W, C] channels-last.
    H-scanner: reshape [B, H, W, C] → [B*W, H, C] → Mamba → reshape back
    W-scanner: reshape [B, H, W, C] → [B*H, W, C] → Mamba → reshape back
    """
    def __init__(self, dim, mlp_ratio=4.0, drop_path=0.0,
                 d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        # 4 Mamba instances: H_fwd, H_bwd, W_fwd, W_bwd
        mamba_kw = dict(d_state=d_state, d_conv=d_conv, expand=expand)
        self.h_fwd = Mamba(d_model=dim, **mamba_kw)
        self.h_bwd = Mamba(d_model=dim, **mamba_kw)
        self.w_fwd = Mamba(d_model=dim, **mamba_kw)
        self.w_bwd = Mamba(d_model=dim, **mamba_kw)
        self.proj  = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = MLP(dim, mlp_ratio)
        self.dp    = DropPath(drop_path) if drop_path > 0 else nn.Identity()
 
    def _bidi(self, fwd, bwd, x):
        """x: [B_eff, L, C] → bidirectional scan, outputs averaged."""
        return fwd(x) #+ bwd(x.flip(1)).flip(1)) * 0.5
 
    def forward(self, x):                                   # [B, H, W, C]
        B, H, W, C = x.shape
        res = x; x = self.norm1(x)
 
        # H-scan: [B*W, H, C]
        h_in  = x.permute(0, 2, 1, 3).contiguous().view(B * W, H, C)
        h_out = self._bidi(self.h_fwd, self.h_bwd, h_in)
        h_out = h_out.view(B, W, H, C).permute(0, 2, 1, 3).contiguous()
 
        # W-scan: [B*H, W, C]
        w_in  = x.view(B * H, W, C)
        w_out = self._bidi(self.w_fwd, self.w_bwd, w_in)
        w_out = w_out.view(B, H, W, C)
 
        x = res + self.dp(self.proj(h_out + w_out))
        return x + self.dp(self.mlp(self.norm2(x)))
 
 
class Downsample2D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.conv = nn.Conv2d(dim, 2 * dim, 2, stride=2)
    def forward(self, x):          # [B, H, W, C]
        return self.conv(self.norm(x).permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
 
 
class MambaEncoder2D(nn.Module):
    """
    4-stage 2D Mamba encoder for ImageNet denoising pre-training.
    Stages 1–4 all use TriCruciMamba2D (no windowed attention).
    Input:  [B, 3, 224, 224]
    Output: bottleneck [B, 8C, 14, 14] + skips [s1, s2, s3, s4]
    """
    def __init__(self, cfg: PretrainConfig):
        super().__init__()
        C, D = cfg.feature_size, cfg.depths
        mamba_kw = dict(d_state=cfg.d_state, d_conv=cfg.d_conv, expand=cfg.mamba_expand)
        dp = torch.linspace(0, cfg.drop_path, sum(D)).tolist()
 
        #self.stem      = nn.Conv2d(cfg.in_channels, C, 2, stride=2)
        self.stem = nn.Sequential(
                        nn.Conv2d(3, C//2, 3, stride=2, padding=1),
                        nn.GELU(),
                        nn.Conv2d(C//2, C, 3, stride=2, padding=1),
                                 )
        self.stem_norm = nn.LayerNorm(C)
 
        s = 0
        self.stage1 = nn.ModuleList(
            [TriCruciMamba2D(C,   cfg.mlp_ratio, dp[s+i], **mamba_kw) for i in range(D[0])])
        self.down1  = Downsample2D(C);   s += D[0]
 
        self.stage2 = nn.ModuleList(
            [TriCruciMamba2D(2*C, cfg.mlp_ratio, dp[s+i], **mamba_kw) for i in range(D[1])])
        self.down2  = Downsample2D(2*C); s += D[1]
 
        self.stage3 = nn.ModuleList(
            [TriCruciMamba2D(4*C, cfg.mlp_ratio, dp[s+i], **mamba_kw) for i in range(D[2])])
        self.down3  = Downsample2D(4*C); s += D[2]
 
        self.stage4 = nn.ModuleList(
            [TriCruciMamba2D(8*C, cfg.mlp_ratio, dp[s+i], **mamba_kw) for i in range(D[3])])
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
 
    def forward(self, x):                                   # [B, 3, 224, 224]
        x  = self.stem(x)                                   # [B, C, 112, 112]
        x  = self.stem_norm(x.permute(0,2,3,1))            # [B, 112, 112, C]
        s1 = x.permute(0,3,1,2).contiguous()               # skip (unused in DAE)
 
        for blk in self.stage1: x = blk(x)
        s2 = x.permute(0,3,1,2).contiguous()               # [B, C, 112, 112]
        x  = self.down1(x)                                   # [B, 56, 56, 2C]
 
        for blk in self.stage2: x = blk(x)
        s3 = x.permute(0,3,1,2).contiguous()               # [B, 2C, 56, 56]
        x  = self.down2(x)
 
        for blk in self.stage3: x = blk(x)
        s4 = x.permute(0,3,1,2).contiguous()               # [B, 4C, 28, 28]
        x  = self.down3(x)
 
        for blk in self.stage4: x = blk(x)
        btk = x.permute(0,3,1,2).contiguous()              # [B, 8C, 14, 14]
        return btk, [s1, s2, s3, s4]
 
 
class DecoderStage2D(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up     = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.refine = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch), nn.GELU())
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.refine(torch.cat([x, skip], dim=1))
 
 
class MambaDenoisingAutoencoder(nn.Module):
    def __init__(self, cfg: PretrainConfig):
        super().__init__()
        C = cfg.feature_size
        self.encoder = MambaEncoder2D(cfg)
        self.dec4 = DecoderStage2D(8*C, 4*C, 4*C)
        self.dec3 = DecoderStage2D(4*C, 2*C, 2*C)
        self.dec2 = DecoderStage2D(2*C,   C,   C)
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(C, C, 2, stride=2),   # 24 -> 48
            nn.GroupNorm(8, C),
            nn.GELU(),

            nn.ConvTranspose2d(C, C, 2, stride=2),   # 48 -> 96
            nn.GroupNorm(8, C),
            nn.GELU(),

            nn.Conv2d(C, C, 3, padding=1, bias=False),
            nn.GroupNorm(8, C),
            nn.GELU(),)
        self.dec1_skip = nn.Sequential(
            nn.Conv2d(C, C, 3, padding=1, bias=False),
            nn.GroupNorm(8, C), nn.GELU())
        self.head = nn.Sequential(
            nn.Conv2d(C, C, 3, padding=1), nn.GELU(),
            nn.Conv2d(C, cfg.in_channels, 1), nn.Sigmoid())
 
    def forward(self, x):
        btk, (s1, s2, s3, s4) = self.encoder(x)
        x = self.dec4(btk, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x)
        return self.head(x)

In [8]:
# %% ═══ Cell 5: Training ════════════════════════════════════════════════
 
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.sum = self.count = 0.0
    def update(self, v, n=1): self.sum += v*n; self.count += n
    @property
    def avg(self): return self.sum / max(self.count, 1)
 
 
def compute_psnr(mse): return 100.0 if mse <= 0 else 10.0 * math.log10(1.0 / mse)
 
 
def pretrain(cfg: PretrainConfig, train_loader, val_loader):
    print("\n" + "═"*60, flush=True)
    print("  Mamba Encoder — ImageNet Denoising Pre-training", flush=True)
    print("═"*60, flush=True)
 
    model = MambaDenoisingAutoencoder(cfg).to(DEVICE)
    n_par = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Parameters : {n_par:.2f}M", flush=True)
 
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler    = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
 
    total_steps  = cfg.num_epochs * len(train_loader)
    warmup_steps = cfg.warmup_epochs * len(train_loader)
    wr = cfg.warmup_lr / cfg.lr; mr = cfg.min_lr / cfg.lr
 
    def lr_lambda(step):
        if step < warmup_steps:
            t = step / max(warmup_steps, 1)
            return wr + t * (1.0 - wr)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return mr + 0.5 * (1.0 + math.cos(math.pi * progress)) * (1.0 - mr)
 
    scheduler  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    best_val   = float("inf")
    no_improve = 0
    history    = []
 
    for epoch in range(1, cfg.num_epochs + 1):
        print(f"\n── Epoch {epoch}/{cfg.num_epochs} ──────────────", flush=True)
 
        # Train
        model.train(); loss_m = AverageMeter(); t0 = time.time()
        for step, (noisy, clean) in enumerate(train_loader):
            noisy = noisy.to(DEVICE, non_blocking=True)
            clean = clean.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                loss = F.mse_loss(model(noisy), clean)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            loss_m.update(loss.item(), noisy.size(0))
            if (step + 1) % cfg.log_every == 0:
                print(f"  [{step+1:5d}/{len(train_loader)}] "
                      f"loss={loss_m.avg:.6f}  "
                      f"lr={optimizer.param_groups[0]['lr']:.2e}  "
                      f"t={time.time()-t0:.1f}s", flush=True)
                t0 = time.time()
 
        # Validate
        model.eval(); vm = AverageMeter()
        with torch.no_grad():
            for noisy, clean in val_loader:
                noisy = noisy.to(DEVICE); clean = clean.to(DEVICE)
                with torch.amp.autocast("cuda", enabled=cfg.use_amp):
                    mse = F.mse_loss(model(noisy), clean)
                vm.update(mse.item(), noisy.size(0))
 
        is_best = vm.avg < best_val
        if is_best: best_val = vm.avg; no_improve = 0
        else:       no_improve += 1
 
        vram = torch.cuda.max_memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        mark = "  ★" if is_best else ""
        print(f"► Ep{epoch:3d}  train={loss_m.avg:.6f}  "
              f"val={vm.avg:.6f}  PSNR={compute_psnr(vm.avg):.2f}dB  "
              f"VRAM={vram:.1f}GB{mark}", flush=True)
 
        history.append({"epoch": epoch, "train_loss": round(loss_m.avg, 6),
                         "val_loss": round(vm.avg, 6), "lr": optimizer.param_groups[0]["lr"]})
 
        if is_best or epoch % cfg.save_every == 0:
            tag = "best" if is_best else f"ep{epoch:03d}"
            # Save encoder-only state dict for 3D inflation
            torch.save({"epoch": epoch,
                         "encoder_state_dict": model.encoder.state_dict(),
                         "feature_size": cfg.feature_size,
                         "depths":       cfg.depths,
                         "d_state":      cfg.d_state,
                         "d_conv":       cfg.d_conv,
                         "mamba_expand": cfg.mamba_expand,
                         "val_loss":     vm.avg},
                        os.path.join(cfg.checkpoint_dir, f"mamba_encoder_{tag}.pth"))
            if is_best:
                print(f"  Encoder saved: mamba_encoder_best.pth", flush=True)
 
        with open(os.path.join(cfg.output_dir, "pretrain_history.json"), "w") as f:
            json.dump(history, f, indent=2)
 
        if no_improve >= cfg.early_stop:
            print(f"\n  Early stop at epoch {epoch}.", flush=True)
            break
 
    print(f"\n  Done. Best val MSE: {best_val:.6f}", flush=True)
    print(f"  Encoder → {cfg.checkpoint_dir}/mamba_encoder_best.pth", flush=True)
    return model, history

In [ ]:
# %% ═══ Cell 6: Visualisation + Entry Point ══════════════════════════════
 
@torch.no_grad()
def save_reconstruction_grid(model, val_loader, cfg, n=4):
    model.eval()
    noisy_b, clean_b = next(iter(val_loader))
    noisy_b = noisy_b[:n].to(DEVICE)
    with torch.amp.autocast("cuda", enabled=cfg.use_amp):
        pred_b = model(noisy_b)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    for i in range(n):
        for j, (img, title) in enumerate([
            (noisy_b, "Noisy"), (pred_b, "Denoised"), (clean_b[:n].to(DEVICE), "Clean")
        ]):
            axes[i, j].imshow(img[i].cpu().float().clamp(0,1).permute(1,2,0).numpy())
            axes[i, j].axis("off")
            if i == 0: axes[i, j].set_title(title)
    plt.tight_layout()
    path = os.path.join(cfg.output_dir, "mamba_reconstruction_grid.png")
    plt.savefig(path, dpi=120); plt.close()
    print(f"  Saved: {path}", flush=True)
 
 
# ── Entry Point ──────────────────────────────────────────────────────────
print("\n" + "═"*60, flush=True)
print("  Mamba ImageNet Pre-training — Entry Point", flush=True)
print("═"*60, flush=True)
 
data_path = Path(cfg.data_dir)
if not data_path.exists():
    print(f"\n⚠  Dataset not found: {cfg.data_dir}", flush=True)
    print("   Sanity check: forward pass on CPU with tiny config", flush=True)
    _c = PretrainConfig(); _c.feature_size = 16; _c.depths = (1,1,1,1)
    _c.d_state = 4; _c.d_conv = 4; _c.mamba_expand = 2
    _m = MambaDenoisingAutoencoder(_c)
    _x = torch.randn(1, 3, 224, 224)
    with torch.no_grad(): _y = _m(_x)
    assert _y.shape == (1, 3, 224, 224), f"Shape error: {_y.shape}"
    print(f"  ✓ {tuple(_x.shape)} → {tuple(_y.shape)}  ({sum(p.numel() for p in _m.parameters())/1e6:.1f}M)", flush=True)
    print("  Add ImageNet dataset and re-run.", flush=True)
    del _m, _x, _y
else:
    train_loader, val_loader = build_dataloaders(cfg)
    model, history           = pretrain(cfg, train_loader, val_loader)
    save_reconstruction_grid(model, val_loader, cfg)
    print("\n  Pre-training complete. Run finetune_mod_a_brats.py next.", flush=True)

In [11]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Modification A: Mamba Encoder + CNN Decoder                       ║
# ║  3D Brain Tumour Segmentation · BraTS-Africa 2025                  ║
# ║  AGENT 2: CODER — Full 9-Cell Implementation                       ║
# ║  H100 80GB · Modal GPU Notebook · PyTorch 2.5.1+cu121              ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# PREREQUISITE: Run pretrain_mamba_imagenet.py first to generate:
#   /root/checkpoints/mamba_encoder_best.pth
#
# CELL EXECUTION ORDER (top to bottom, no edits required):
#   Cell 1 : Install dependencies
#   Cell 2 : Imports + Config
#   Cell 3 : BraTS-Africa data pipeline
#   Cell 4 : Model (TriCruciMamba3D encoder + CNN decoder + weight inflation)
#   Cell 5 : Loss (DiceCELoss) + metrics (Dice, HD95)
#   Cell 6 : Training loop (Phase 1 frozen + Phase 2 unfrozen)
#   Cell 7 : Evaluation (sliding-window, full-volume Dice + HD95)
#   Cell 8 : Visualisation (overlays, metric charts, training curves)
#   Cell 9 : Sanity checks + entry point (runs all cells in sequence)
#
# BraTS-Africa label convention (detected at runtime from np.unique):
#   Old BraTS: ET=4, TC={1,4}, WT={1,2,4}
#   BraTS 2023: ET=3, TC={1,3}, WT={1,2,3}
# ─────────────────────────────────────────────────────────────────────────

In [32]:
# %% Cell 1: Dependencies
# FIX-OOM-1: must be set before any torch import
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
 
import subprocess, sys, re, importlib
 
def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + list(pkgs))
 
print("Cell 1: setting up environment...", flush=True)
 
deps = [
    ("torch",        None),
    ("torchvision",  None),
    ("monai",        "monai[all]==1.3.2"),
    ("nibabel",      "nibabel"),
    ("timm",         "timm==0.9.12"),
    ("scipy",        "scipy"),
    ("matplotlib",   "matplotlib"),
    ("sklearn",      "scikit-learn"),
]
for mod, pkg in deps:
    try:
        __import__(mod)
        print(f"  {mod}: already installed", flush=True)
    except ImportError:
        print(f"  Installing {pkg or mod} ...", flush=True)
        _pip(pkg or mod)
        print(f"  {mod}: installed", flush=True)
 
_mamba_installed = False
try:
    from mamba_ssm import Mamba
    _mamba_installed = True
    print("mamba-ssm: CUDA kernel installed", flush=True)
except ImportError:
    print("mamba-ssm: using pure-PyTorch shim", flush=True)
 
if not _mamba_installed:
    import torch as _t
    import torch.nn as _nn
    import torch.nn.functional as _F
    import types as _types, sys as _sys
 
    class Mamba(_nn.Module):
        """
        Pure-PyTorch Mamba SSM.
        FIX-OOM-2: per-step dA_i/dB_i instead of precomputing for all L.
          Before: dA shape [B_eff, L, d_inner, d_state] = 3.2 GB at Stage 1.
          After:  dA_i shape [B_eff, d_inner, d_state]  = 50 MB per step.
        """
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2, **kwargs):
            super().__init__()
            self.d_model  = d_model
            self.d_state  = d_state
            self.d_inner  = d_model * expand
            dt_rank       = max(1, d_model // 16)
            self.in_proj  = _nn.Linear(d_model, self.d_inner * 2, bias=False)
            self.conv1d   = _nn.Conv1d(self.d_inner, self.d_inner,
                                        kernel_size=d_conv, padding=d_conv - 1,
                                        groups=self.d_inner, bias=True)
            self.x_proj   = _nn.Linear(self.d_inner, dt_rank + d_state * 2, bias=False)
            self.dt_proj  = _nn.Linear(dt_rank, self.d_inner, bias=True)
            A = _t.arange(1, d_state + 1, dtype=_t.float32).unsqueeze(0).repeat(self.d_inner, 1)
            self.A_log    = _nn.Parameter(_t.log(A))
            self.D        = _nn.Parameter(_t.ones(self.d_inner))
            self.out_proj = _nn.Linear(self.d_inner, d_model, bias=False)
            self.norm     = _nn.LayerNorm(self.d_inner)
 
        def forward(self, x):
            B, L, _ = x.shape
            xz    = self.in_proj(x)
            x_br  = xz[..., :self.d_inner]
            z     = xz[..., self.d_inner:]
            x_br  = _F.silu(self.conv1d(x_br.transpose(1, 2))[..., :L].transpose(1, 2))
            A       = -_t.exp(self.A_log.float())
            dBx     = self.x_proj(x_br)
            dt_rank = self.dt_proj.in_features
            dt      = _F.softplus(self.dt_proj(dBx[..., :dt_rank]))
            B_ssm   = dBx[..., dt_rank:dt_rank + self.d_state]
            C       = dBx[..., dt_rank + self.d_state:]
            # FIX-OOM-2: per-step computation
            h  = _t.zeros(B, self.d_inner, self.d_state, dtype=_t.float32, device=x.device)
            ys = []
            for i in range(L):
                dt_i  = dt[:, i]
                dA_i  = _t.exp(dt_i.unsqueeze(-1) * A)
                dB_i  = dt_i.unsqueeze(-1) * B_ssm[:, i].unsqueeze(1)
                h     = dA_i * h + dB_i * x_br[:, i].unsqueeze(-1).float()
                y     = (h * C[:, i].float().unsqueeze(1)).sum(-1)
                ys.append(y)
            y = _t.stack(ys, dim=1) + self.D * x_br
            y = self.norm(y) * _F.silu(z)
            return self.out_proj(y)
 
    _mod       = _types.ModuleType("mamba_ssm")
    _mod.Mamba = Mamba
    _sys.modules["mamba_ssm"] = _mod
    print("  Mamba shim registered (per-step dA/dB, OOM-safe)", flush=True)
 
import torch
assert torch.cuda.is_available(), "CUDA required"
print(f"PyTorch : {torch.__version__}", flush=True)
print(f"GPU     : {torch.cuda.get_device_name(0)} "
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)", flush=True)
print("Cell 1 complete", flush=True)

In [33]:
# %% Cell 2: Imports + Config
import math, time, json, random, warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
 
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.amp
import torch.utils.checkpoint
from torch.utils.data import DataLoader, Dataset
 
from mamba_ssm import Mamba
from timm.models.layers import trunc_normal_, DropPath
 
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric, HausdorffDistanceMetric
 
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
 
warnings.filterwarnings("ignore", category=UserWarning)
 
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda")
 
MODALITIES  = ["t1n", "t1c", "t2w", "t2f"]
CLASS_NAMES = ["ET", "TC", "WT"]
ET_LABEL: Optional[int] = None
 
 
@dataclass
class Config:
    data_root:           str   = "/root/datasets/brats-africa/BraTS-Africa Dataset/BraTS-Africa"
    data_subdirs:        List  = field(default_factory=lambda: ["51_OtherNeoplasms", "95_Glioma"])
    mamba_pretrain_ckpt: str   = "/root/checkpoints/mamba_encoder_best.pth"
    checkpoint_dir:      str   = "/root/checkpoints/mod_a"
    output_dir:          str   = "/root/outputs/mod_a"
    in_channels:         int   = 4
    num_classes:         int   = 3
    feature_size:        int   = 48
    depths:              Tuple = (2, 2, 2, 2)
    mlp_ratio:           float = 4.0
    drop_path:           float = 0.1
    d_state:             int   = 16
    d_conv:              int   = 4
    mamba_expand:        int   = 2
    patch_size:          Tuple = (128, 128, 128)
    phase1_epochs:       int   = 20
    phase2_epochs:       int   = 30
    decoder_lr:          float = 3e-4
    encoder_lr:          float = 3e-5
    min_lr:              float = 1e-6
    warmup_epochs:       int   = 3
    weight_decay:        float = 0.05
    grad_clip:           float = 1.0
    batch_size:          int   = 2
    use_amp:             bool  = True
    save_every:          int   = 5
    log_every:           int   = 20
    val_fraction:        float = 0.15
    test_fraction:       float = 0.10
    sw_overlap:          float = 0.5
    sw_batch_size:       int   = 2
 
    def __post_init__(self):
        os.makedirs(self.checkpoint_dir, exist_ok=True)
        os.makedirs(self.output_dir, exist_ok=True)
 
 
cfg = Config()
print(f"Config: C={cfg.feature_size} d_state={cfg.d_state} "
      f"P1={cfg.phase1_epochs}ep P2={cfg.phase2_epochs}ep", flush=True)

In [34]:
# %% Cell 3: BraTS-Africa Data Pipeline
 
def detect_label_convention(seg_path: Path) -> int:
    global ET_LABEL
    if ET_LABEL is not None:
        return ET_LABEL
    seg    = nib.load(str(seg_path)).get_fdata()
    unique = [int(v) for v in np.unique(seg)]
    if 4 in unique:
        ET_LABEL = 4
        print(f"  Label convention: ET=4 (old BraTS) labels={unique}", flush=True)
    elif 3 in unique:
        ET_LABEL = 3
        print(f"  Label convention: ET=3 (BraTS 2023) labels={unique}", flush=True)
    else:
        ET_LABEL = 3
        print(f"  ET label ambiguous: {unique} defaulting ET=3", flush=True)
    return ET_LABEL
 
 
def discover_cases(cfg: Config) -> List[Dict]:
    root = Path(cfg.data_root)
    assert root.exists(), f"Data root not found: {root}"
    cases: List[Dict] = []
    for sdir_name in cfg.data_subdirs:
        sdir = root / sdir_name
        if not sdir.exists():
            print(f"  Subdir missing: {sdir}", flush=True); continue
        case_dirs = sorted([d for d in sdir.iterdir() if d.is_dir()])
        if case_dirs:
            for cdir in case_dirs:
                cid = cdir.name; mods, seg_path = {}, None
                for ext in [".nii", ".nii.gz"]:
                    for m in MODALITIES:
                        p = cdir / f"{cid}-{m}{ext}"
                        if p.exists() and p.stat().st_size > 0 and m not in mods:
                            mods[m] = p
                    sp = cdir / f"{cid}-seg{ext}"
                    if sp.exists() and sp.stat().st_size > 0 and seg_path is None:
                        seg_path = sp
                if len(mods) == 4 and seg_path is not None:
                    cases.append({"case_id": cid, "modalities": mods,
                                  "seg": seg_path, "subdir": sdir_name})
                else:
                    print(f"  Incomplete: {cid}", flush=True)
        else:
            all_files = [f for f in list(sdir.glob("*.nii")) + list(sdir.glob("*.nii.gz"))
                         if f.stat().st_size > 0]
            by_case: Dict[str, Dict] = {}
            for f in all_files:
                stem = f.name.split(".nii")[0]
                for tag in MODALITIES + ["seg"]:
                    if stem.endswith(f"-{tag}"):
                        by_case.setdefault(stem[:-len(f"-{tag}")], {})[tag] = f
            for cid, files in sorted(by_case.items()):
                if all(m in files for m in MODALITIES) and "seg" in files:
                    cases.append({"case_id": cid,
                                  "modalities": {m: files[m] for m in MODALITIES},
                                  "seg": files["seg"], "subdir": sdir_name})
    assert len(cases) > 0, "No complete cases found."
    detect_label_convention(cases[0]["seg"])
    print(f"Discovered {len(cases)} complete cases.", flush=True)
    return cases
 
 
def split_cases(cases: List[Dict], cfg: Config) -> Tuple[List, List, List]:
    rng = np.random.RandomState(SEED)
    by_sub: Dict[str, List] = {}
    for c in cases: by_sub.setdefault(c["subdir"], []).append(c)
    train, val, test = [], [], []
    for group in by_sub.values():
        idx   = rng.permutation(len(group))
        n_tst = max(1, int(len(group) * cfg.test_fraction))
        n_val = max(1, int(len(group) * cfg.val_fraction))
        test  += [group[i] for i in idx[:n_tst]]
        val   += [group[i] for i in idx[n_tst:n_tst+n_val]]
        train += [group[i] for i in idx[n_tst+n_val:]]
    print(f"Split: train={len(train)} val={len(val)} test={len(test)}", flush=True)
    return train, val, test
 
 
def load_nifti(path: Path) -> np.ndarray:
    try:
        return nib.load(str(path)).get_fdata(dtype=np.float32)
    except Exception as exc:
        size = path.stat().st_size if path.exists() else "MISSING"
        raise RuntimeError(f"Cannot load NIfTI {path} size={size}: {exc}") from exc
 
 
def zscore_normalize(vol: np.ndarray) -> np.ndarray:
    mask = vol > 0
    if not mask.any(): return vol
    mu, sigma = vol[mask].mean(), vol[mask].std()
    out = np.zeros_like(vol)
    out[mask] = (vol[mask] - mu) / (sigma + 1e-8)
    return out
 
 
def labels_to_channels(seg: np.ndarray) -> np.ndarray:
    assert ET_LABEL is not None
    et = (seg == ET_LABEL).astype(np.float32)
    tc = ((seg == 1) | (seg == ET_LABEL)).astype(np.float32)
    wt = (seg > 0).astype(np.float32)
    return np.stack([et, tc, wt], axis=0)
 
 
def pad_to_min(arr: np.ndarray, min_shape: Tuple) -> np.ndarray:
    if arr.ndim == 4:
        D, H, W = arr.shape[1:]
        pd = max(0, min_shape[0]-D); ph = max(0, min_shape[1]-H); pw = max(0, min_shape[2]-W)
        if pd or ph or pw: arr = np.pad(arr, ((0,0),(0,pd),(0,ph),(0,pw)))
    else:
        D, H, W = arr.shape
        pd = max(0, min_shape[0]-D); ph = max(0, min_shape[1]-H); pw = max(0, min_shape[2]-W)
        if pd or ph or pw: arr = np.pad(arr, ((0,pd),(0,ph),(0,pw)))
    return arr
 
 
def random_crop_3d(image, label, size):
    _, D, H, W = image.shape; cd, ch, cw = size
    d0 = random.randint(0, max(0, D-cd))
    h0 = random.randint(0, max(0, H-ch))
    w0 = random.randint(0, max(0, W-cw))
    return (image[:, d0:d0+cd, h0:h0+ch, w0:w0+cw],
            label[:, d0:d0+cd, h0:h0+ch, w0:w0+cw])
 
 
def center_crop_3d(image, label, size):
    image = pad_to_min(image, size); label = pad_to_min(label, size)
    _, D, H, W = image.shape; cd, ch, cw = size
    d0 = (D-cd)//2; h0 = (H-ch)//2; w0 = (W-cw)//2
    return (image[:, d0:d0+cd, h0:h0+ch, w0:w0+cw],
            label[:, d0:d0+cd, h0:h0+ch, w0:w0+cw])
 
 
class BraTSPatchDataset(Dataset):
    def __init__(self, cases, cfg, training=True):
        self.cases = cases; self.cfg = cfg; self.training = training
    def __len__(self): return len(self.cases)
    def __getitem__(self, idx):
        for attempt in range(len(self.cases)):
            try:
                case = self.cases[(idx + attempt) % len(self.cases)]
                vols = [zscore_normalize(load_nifti(case["modalities"][m])) for m in MODALITIES]
                image = np.stack(vols, axis=0).astype(np.float32)
                label = labels_to_channels(load_nifti(case["seg"]))
                image = pad_to_min(image, self.cfg.patch_size)
                label = pad_to_min(label, self.cfg.patch_size)
                if self.training:
                    image, label = random_crop_3d(image, label, self.cfg.patch_size)
                    for ax in [1, 2, 3]:
                        if random.random() < 0.5:
                            image = np.flip(image, axis=ax).copy()
                            label = np.flip(label, axis=ax).copy()
                    shift = np.random.uniform(-0.1, 0.1, (4,1,1,1)).astype(np.float32)
                    scale = np.random.uniform( 0.9, 1.1, (4,1,1,1)).astype(np.float32)
                    image = image * scale + shift
                else:
                    image, label = center_crop_3d(image, label, self.cfg.patch_size)
                return torch.from_numpy(image), torch.from_numpy(label)
            except Exception as exc:
                cid = self.cases[(idx+attempt) % len(self.cases)]["case_id"]
                print(f"  Patch [{attempt+1}] skipping {cid}: {exc}", flush=True)
                if attempt == len(self.cases) - 1:
                    raise RuntimeError(f"All cases failed idx={idx}: {exc}") from exc
 
 
class BraTSVolumeDataset(Dataset):
    def __init__(self, cases): self.cases = cases
    def __len__(self): return len(self.cases)
    def __getitem__(self, idx):
        for attempt in range(len(self.cases)):
            try:
                case  = self.cases[(idx + attempt) % len(self.cases)]
                vols  = [zscore_normalize(load_nifti(case["modalities"][m])) for m in MODALITIES]
                image = torch.from_numpy(np.stack(vols, axis=0).astype(np.float32))
                label = torch.from_numpy(labels_to_channels(load_nifti(case["seg"])))
                return image, label, case["case_id"]
            except Exception as exc:
                cid = self.cases[(idx+attempt) % len(self.cases)]["case_id"]
                print(f"  Volume [{attempt+1}] skipping {cid}: {exc}", flush=True)
                if attempt == len(self.cases) - 1:
                    raise RuntimeError(f"All cases failed idx={idx}: {exc}") from exc
 
 
def build_loaders(cfg: Config):
    cases = discover_cases(cfg)
    train_cases, val_cases, test_cases = split_cases(cases, cfg)
    kw = dict(num_workers=0, pin_memory=True)
    train_loader   = DataLoader(BraTSPatchDataset(train_cases, cfg, True),
                                cfg.batch_size, shuffle=True, drop_last=True, **kw)
    val_loader     = DataLoader(BraTSPatchDataset(val_cases, cfg, False),
                                cfg.batch_size, shuffle=False, **kw)
    val_vol_loader = DataLoader(BraTSVolumeDataset(val_cases), 1, shuffle=False, **kw)
    test_loader    = DataLoader(BraTSVolumeDataset(test_cases), 1, shuffle=False, **kw)
    print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)}", flush=True)
    return train_loader, val_loader, val_vol_loader, test_loader, val_cases, test_cases

In [35]:
# %% Cell 4: Model
 
class MLP(nn.Module):
    def __init__(self, dim, ratio=4.0):
        super().__init__()
        h = int(dim * ratio)
        self.net = nn.Sequential(nn.Linear(dim, h), nn.GELU(), nn.Linear(h, dim))
    def forward(self, x): return self.net(x)
 
 
class TriCruciMamba3D(nn.Module):
    """3D TriCruci block. _bidi casts float32 to prevent AMP NaN."""
    def __init__(self, dim, mlp_ratio=4.0, drop_path=0.0, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        kw = dict(d_state=d_state, d_conv=d_conv, expand=expand)
        self.d_fwd = Mamba(d_model=dim, **kw); self.d_bwd = Mamba(d_model=dim, **kw)
        self.h_fwd = Mamba(d_model=dim, **kw); self.h_bwd = Mamba(d_model=dim, **kw)
        self.w_fwd = Mamba(d_model=dim, **kw); self.w_bwd = Mamba(d_model=dim, **kw)
        self.proj  = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = MLP(dim, mlp_ratio)
        self.dp    = DropPath(drop_path) if drop_path > 0 else nn.Identity()
 
    def _bidi(self, fwd, bwd, x):
        dtype = x.dtype; x = x.float()
        return ((fwd(x) + bwd(x.flip(1)).flip(1)) * 0.5).to(dtype)
 
    def forward(self, x):
        B, D, H, W, C = x.shape
        res = x; x = self.norm1(x)
        d_in  = x.permute(0,2,3,1,4).contiguous().view(B*H*W, D, C)
        d_out = self._bidi(self.d_fwd, self.d_bwd, d_in)
        d_out = d_out.view(B,H,W,D,C).permute(0,3,1,2,4).contiguous()
        h_in  = x.permute(0,1,3,2,4).contiguous().view(B*D*W, H, C)
        h_out = self._bidi(self.h_fwd, self.h_bwd, h_in)
        h_out = h_out.view(B,D,W,H,C).permute(0,1,3,2,4).contiguous()
        w_in  = x.contiguous().view(B*D*H, W, C)
        w_out = self._bidi(self.w_fwd, self.w_bwd, w_in)
        w_out = w_out.view(B,D,H,W,C)
        x = res + self.dp(self.proj(d_out + h_out + w_out))
        return x + self.dp(self.mlp(self.norm2(x)))
 
 
class Downsample3D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.conv = nn.Conv3d(dim, 2*dim, 2, stride=2)
    def forward(self, x):
        return self.conv(self.norm(x).permute(0,4,1,2,3)).permute(0,2,3,4,1)
 
 
class MambaEncoder3D(nn.Module):
    """
    FIX-OOM-4: gradient checkpointing at Stages 1+2.
    Phase 2 BPTT at Stage 1 (64^3): 38.4 GB without checkpointing -> OOM.
    With checkpointing: ~50 MB per block via recomputation.
    """
    def __init__(self, cfg: Config):
        super().__init__()
        C, D = cfg.feature_size, cfg.depths
        kw = dict(d_state=cfg.d_state, d_conv=cfg.d_conv, expand=cfg.mamba_expand)
        dp = torch.linspace(0, cfg.drop_path, sum(D)).tolist()
        self.stem      = nn.Conv3d(cfg.in_channels, C, 2, stride=2)
        self.stem_norm = nn.LayerNorm(C)
        s = 0
        self.stage1 = nn.ModuleList([TriCruciMamba3D(C,   cfg.mlp_ratio, dp[s+i], **kw) for i in range(D[0])])
        self.down1  = Downsample3D(C);   s += D[0]
        self.stage2 = nn.ModuleList([TriCruciMamba3D(2*C, cfg.mlp_ratio, dp[s+i], **kw) for i in range(D[1])])
        self.down2  = Downsample3D(2*C); s += D[1]
        self.stage3 = nn.ModuleList([TriCruciMamba3D(4*C, cfg.mlp_ratio, dp[s+i], **kw) for i in range(D[2])])
        self.down3  = Downsample3D(4*C); s += D[2]
        self.stage4 = nn.ModuleList([TriCruciMamba3D(8*C, cfg.mlp_ratio, dp[s+i], **kw) for i in range(D[3])])
        self.dims           = [C, 2*C, 4*C, 8*C]
        self.use_checkpoint = False
        self._init_weights()
 
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.Linear)):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
 
    def enable_checkpointing(self, enabled: bool = True):
        """Call before Phase 2. Trades recomputation for BPTT memory at Stages 1-2."""
        self.use_checkpoint = enabled
        print(f"  Encoder gradient checkpointing: {'ON' if enabled else 'OFF'}", flush=True)
 
    def _run_block(self, blk, x):
        if self.use_checkpoint and x.requires_grad:
            return torch.utils.checkpoint.checkpoint(blk, x, use_reentrant=False)
        return blk(x)
 
    def forward(self, x):
        x  = self.stem(x)
        x  = self.stem_norm(x.permute(0,2,3,4,1))
        s1 = x.permute(0,4,1,2,3).contiguous()
        # Stages 1-2: checkpointed in Phase 2
        for blk in self.stage1: x = self._run_block(blk, x)
        s2 = x.permute(0,4,1,2,3).contiguous(); x = self.down1(x)
        for blk in self.stage2: x = self._run_block(blk, x)
        s3 = x.permute(0,4,1,2,3).contiguous(); x = self.down2(x)
        # Stages 3-4: not checkpointed (B_eff 64x smaller)
        for blk in self.stage3: x = blk(x)
        s4 = x.permute(0,4,1,2,3).contiguous(); x = self.down3(x)
        for blk in self.stage4: x = blk(x)
        return x.permute(0,4,1,2,3).contiguous(), [s1, s2, s3, s4]
 
 
class DecoderStage3D(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up     = nn.ConvTranspose3d(in_ch, out_ch, 2, stride=2)
        self.refine = nn.Sequential(
            nn.Conv3d(out_ch+skip_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch), nn.GELU(),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_ch), nn.GELU())
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-3:] != skip.shape[-3:]:
            x = F.interpolate(x, skip.shape[-3:], mode="trilinear", align_corners=False)
        return self.refine(torch.cat([x, skip], dim=1))
 
 
class CNNDecoder3D(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        C = cfg.feature_size
        self.dec4 = DecoderStage3D(8*C, 4*C, 4*C)
        self.dec3 = DecoderStage3D(4*C, 2*C, 2*C)
        self.dec2 = DecoderStage3D(2*C,   C,   C)
        self.dec1 = nn.Sequential(
            nn.ConvTranspose3d(C, C, 2, stride=2),
            nn.GroupNorm(8, C), nn.GELU(),
            nn.Conv3d(C, C, 3, padding=1, bias=False),
            nn.GroupNorm(8, C), nn.GELU())
    def forward(self, btk, skips):
        _s1, s2, s3, s4 = skips
        x = self.dec4(btk, s4); x = self.dec3(x, s3)
        x = self.dec2(x, s2);   return self.dec1(x)
 
 
class MambaUNETR3D(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        C = cfg.feature_size
        self.encoder = MambaEncoder3D(cfg)
        self.decoder = CNNDecoder3D(cfg)
        self.head = nn.Sequential(nn.Conv3d(C, C, 3, padding=1), nn.GELU(),
                                   nn.Conv3d(C, cfg.num_classes, 1))
        for m in self.head.modules():
            if isinstance(m, nn.Conv3d):
                trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        btk, skips = self.encoder(x)
        return self.head(self.decoder(btk, skips))
 
 
def _inflate_conv(key, w2d, target_shape):
    if w2d.dim() != 4: return None
    kd = target_shape[2]
    if "stem.weight" in key:
        w_avg = w2d.mean(dim=1, keepdim=True)
        return w_avg.repeat(1,4,1,1).unsqueeze(2).repeat(1,1,kd,1,1) / kd
    return w2d.unsqueeze(2).repeat(1,1,kd,1,1) / kd
 
 
def load_mamba_2d_to_3d(model_3d, ckpt_path):
    assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    sd2d = ckpt["encoder_state_dict"]; sd3d = model_3d.encoder.state_dict()
    new_sd = {}; skipped = []; inflated = []; direct = []; warm = []
    for key, w3d in sd3d.items():
        src = key.replace(".d_fwd.", ".h_fwd.").replace(".d_bwd.", ".h_bwd.")
        is_warm = src != key
        if src not in sd2d: skipped.append(key); continue
        w2d = sd2d[src]
        if w2d.shape == w3d.shape:
            new_sd[key] = w2d.clone(); (warm if is_warm else direct).append(key)
        elif w2d.dim() == 4:
            wi = _inflate_conv(key, w2d, w3d.shape)
            if wi is not None and wi.shape == w3d.shape:
                new_sd[key] = wi; inflated.append(key)
            else: skipped.append(key)
        else: skipped.append(key)
    for k, v in new_sd.items():
        assert v.shape == sd3d[k].shape
    missing, _ = model_3d.encoder.load_state_dict(new_sd, strict=False)
    print(f"  direct={len(direct)} inflated={len(inflated)} warm={len(warm)} skipped={len(skipped)}",
          flush=True)
    if missing: print(f"  Missing: {missing[:3]}", flush=True)
    return missing
 
 
def build_model(cfg: Config) -> MambaUNETR3D:
    print("\n--- Building MambaUNETR3D ---", flush=True)
    model = MambaUNETR3D(cfg).to(DEVICE)
    n_enc  = sum(p.numel() for p in model.encoder.parameters()) / 1e6
    n_dec  = sum(p.numel() for p in model.decoder.parameters()) / 1e6
    n_head = sum(p.numel() for p in model.head.parameters()) / 1e6
    print(f"  Enc:{n_enc:.1f}M Dec:{n_dec:.1f}M Head:{n_head:.1f}M "
          f"Total:{n_enc+n_dec+n_head:.1f}M", flush=True)
    if Path(cfg.mamba_pretrain_ckpt).exists():
        print(f"  Inflating: {cfg.mamba_pretrain_ckpt}", flush=True)
        load_mamba_2d_to_3d(model, cfg.mamba_pretrain_ckpt)
    else:
        print(f"  Pretrain ckpt missing — random init.", flush=True)
    return model

In [36]:
# %% Cell 5: Loss + Metrics
 
criterion = DiceCELoss(
    include_background=True, to_onehot_y=False, sigmoid=True,
    squared_pred=True, smooth_nr=1.0, smooth_dr=1.0, reduction="mean")
 
dice_metric = DiceMetric(include_background=True, reduction="mean_batch", get_not_nans=True)
hd_metric   = HausdorffDistanceMetric(include_background=True, percentile=95,
                                       directed=False, reduction="mean_batch", get_not_nans=True)
 
 
@torch.no_grad()
def compute_patch_dice(logits, labels):
    try:
        dice_metric(y_pred=(torch.sigmoid(logits) > 0.5).long(), y=labels.long())
        scores, _ = dice_metric.aggregate()
        return scores
    finally:
        dice_metric.reset()
 
 
@torch.no_grad()
def compute_volume_metrics(logits, labels):
    if torch.isnan(logits).any():
        n = int(torch.isnan(logits).sum())
        raise RuntimeError(f"NaN in logits ({n}/{logits.numel()})")
    pred_bin = (torch.sigmoid(logits) > 0.5).long()
    try:
        dice_metric(y_pred=pred_bin, y=labels.long())
        hd_metric(y_pred=pred_bin, y=labels.long())
        dice_s, _ = dice_metric.aggregate()
        hd95_s, _ = hd_metric.aggregate()
        return dice_s, hd95_s
    finally:
        dice_metric.reset(); hd_metric.reset()

In [37]:
# %% Cell 6: Training Loop
 
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.sum = self.count = 0.0
    def update(self, v, n=1): self.sum += v*n; self.count += n
    @property
    def avg(self): return self.sum / max(self.count, 1)
 
 
def make_scheduler(optimizer, warmup_ep, total_ep, steps_per_ep, min_lr_ratio):
    ws = warmup_ep * steps_per_ep; ts = total_ep * steps_per_ep
    def fn(step):
        if step < ws: return 0.01 + (step/max(ws,1))*0.99
        p = (step-ws)/max(ts-ws,1)
        return min_lr_ratio + 0.5*(1+math.cos(math.pi*p))*(1-min_lr_ratio)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)
 
 
def train_one_epoch(model, loader, optimizer, scheduler, scaler, cfg, epoch):
    model.train(); loss_m = AverageMeter(); t0 = time.time()
    for step, (image, label) in enumerate(loader):
        image = image.to(DEVICE, non_blocking=True)
        label = label.to(DEVICE, non_blocking=True)
 
        # FIX-OOM-3: zero_grad BEFORE forward releases previous grad buffers (~4 GB)
        optimizer.zero_grad(set_to_none=True)
 
        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            loss = criterion(model(image), label)
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer); scaler.update(); scheduler.step()
 
        lv = loss.item()
        if not math.isfinite(lv):
            print(f"  Non-finite loss={lv} step={step+1}. Check float32 cast.", flush=True)
        else:
            loss_m.update(lv, image.size(0))
 
        if (step + 1) % cfg.log_every == 0:
            print(f"  [Ep{epoch} {step+1}/{len(loader)}] "
                  f"loss={loss_m.avg:.4f} lr={optimizer.param_groups[0]['lr']:.2e} "
                  f"t={time.time()-t0:.1f}s", flush=True)
            t0 = time.time()
    return loss_m.avg
 
 
@torch.no_grad()
def quick_validate(model, loader, cfg):
    model.eval(); dl = []
    for image, label in loader:
        with torch.amp.autocast("cuda", enabled=cfg.use_amp):
            dl.append(compute_patch_dice(model(image.to(DEVICE)), label.to(DEVICE)).cpu())
    return torch.stack(dl).mean(0)
 
 
def run_training(model, train_loader, val_loader, cfg: Config):
    """
    FIX-OOM-5: before Phase 2:
      1. enable_checkpointing(True) — prevents 38 GB BPTT at Stage 1 (64^3)
      2. torch.cuda.empty_cache()   — frees Phase 1 optimizer residuals
         before Phase 2 AdamW creates new momentum/variance buffers
    Both MUST happen before model.encoder.requires_grad_(True).
    """
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    best_dice = 0.0
    history: Dict[str, List] = {k: [] for k in
        ["train_loss","val_dice_et","val_dice_tc","val_dice_wt","val_dice_mean","lr","phase"]}
 
    for phase, n_epochs, make_opt in [
        (1, cfg.phase1_epochs, lambda m: torch.optim.AdamW(
            list(m.decoder.parameters()) + list(m.head.parameters()),
            lr=cfg.decoder_lr, weight_decay=cfg.weight_decay)),
        (2, cfg.phase2_epochs, lambda m: torch.optim.AdamW([
            {"params": m.encoder.parameters(), "lr": cfg.encoder_lr},
            {"params": list(m.decoder.parameters()) + list(m.head.parameters()),
             "lr": cfg.decoder_lr}], weight_decay=cfg.weight_decay)),
    ]:
        if n_epochs == 0:
            print(f"\n  Phase {phase} skipped.", flush=True); continue
 
        print(f"\n{'='*60}", flush=True)
        print(f"  Phase {phase} encoder {'FROZEN' if phase==1 else 'UNFROZEN'} ({n_epochs} ep)",
              flush=True)
        print(f"{'='*60}", flush=True)
 
        if phase == 2:
            # FIX-OOM-5: MUST happen before requires_grad_(True)
            model.encoder.enable_checkpointing(True)
            torch.cuda.empty_cache()
            print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated "
                  f"{torch.cuda.memory_reserved()/1e9:.1f} GB reserved", flush=True)
 
        model.encoder.requires_grad_(phase == 2)
        optimizer = make_opt(model)
        _base     = min(g["lr"] for g in optimizer.param_groups)
        scheduler = make_scheduler(optimizer, cfg.warmup_epochs, n_epochs,
                                    len(train_loader), cfg.min_lr / _base)
 
        for epoch in range(1, n_epochs + 1):
            ep_abs = (phase-1)*cfg.phase1_epochs + epoch
            print(f"\n-- Ph{phase} Ep{epoch}/{n_epochs} (abs {ep_abs}) --", flush=True)
            train_loss = train_one_epoch(model, train_loader, optimizer,
                                          scheduler, scaler, cfg, ep_abs)
            val_dice   = quick_validate(model, val_loader, cfg)
            mean_dice  = val_dice.mean().item()
            print(f"Ep{epoch:3d} loss={train_loss:.4f} ET={val_dice[0]:.3f} "
                  f"TC={val_dice[1]:.3f} WT={val_dice[2]:.3f} mean={mean_dice:.3f} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}", flush=True)
 
            is_best = mean_dice > best_dice
            if is_best: best_dice = mean_dice
            history["train_loss"].append(round(train_loss,4))
            history["val_dice_et"].append(round(val_dice[0].item(),4))
            history["val_dice_tc"].append(round(val_dice[1].item(),4))
            history["val_dice_wt"].append(round(val_dice[2].item(),4))
            history["val_dice_mean"].append(round(mean_dice,4))
            history["lr"].append(optimizer.param_groups[0]["lr"])
            history["phase"].append(phase)
 
            if is_best or epoch % cfg.save_every == 0:
                tag  = "best" if is_best else f"ph{phase}_ep{epoch:03d}"
                ckpt = {"epoch": ep_abs, "phase": phase,
                         "model_state_dict": model.state_dict(),
                         "val_dice": val_dice.tolist(), "best_dice": best_dice,
                         "config": cfg.__dict__}
                path = os.path.join(cfg.checkpoint_dir, f"modA_{tag}.pth")
                torch.save(ckpt, path)
                if is_best: print(f"  NEW BEST {mean_dice:.4f} -> {path}", flush=True)
 
            with open(os.path.join(cfg.output_dir, "train_history.json"), "w") as f:
                json.dump(history, f, indent=2)
 
    last = {"epoch": cfg.phase1_epochs+cfg.phase2_epochs, "phase": 2,
             "model_state_dict": model.state_dict(),
             "val_dice": [], "best_dice": best_dice, "config": cfg.__dict__}
    torch.save(last, os.path.join(cfg.checkpoint_dir, "modA_epoch_last.pth"))
    print(f"  modA_epoch_last.pth saved. Best Dice: {best_dice:.4f}", flush=True)
    return model, history

In [38]:
# %% Cell 7: Evaluation
 
@torch.no_grad()
def evaluate_full_volumes(model, vol_loader, cfg: Config, split_name="val"):
    model.eval(); all_dice = []; all_hd95 = []
    for image, label, case_id in vol_loader:
        image = image.to(DEVICE); label = label.to(DEVICE)
        logits = sliding_window_inference(
            inputs=image, roi_size=cfg.patch_size, sw_batch_size=cfg.sw_batch_size,
            predictor=model, overlap=cfg.sw_overlap, mode="gaussian")
        dice, hd95 = compute_volume_metrics(logits, label)
        all_dice.append(dice.cpu()); all_hd95.append(hd95.cpu())
        print(f"  [{split_name}] {case_id[0]:30s} "
              f"Dice ET={dice[0]:.3f} TC={dice[1]:.3f} WT={dice[2]:.3f} "
              f"HD95 ET={hd95[0]:.1f} TC={hd95[1]:.1f} WT={hd95[2]:.1f}", flush=True)
    if not all_dice:
        empty = np.full((0, len(CLASS_NAMES)), np.nan)
        return {}, empty, empty
    dice_mat = torch.stack(all_dice).numpy()
    hd95_mat = torch.stack(all_hd95).numpy()
    results = {}
    for ci, cls in enumerate(CLASS_NAMES):
        d_mean = np.nanmean(dice_mat[:,ci]); d_std = np.nanstd(dice_mat[:,ci])
        h_mean = np.nanmean(hd95_mat[:,ci]); h_std = np.nanstd(hd95_mat[:,ci])
        n_nan  = int(np.isnan(hd95_mat[:,ci]).sum())
        results[cls] = {"dice_mean": float(d_mean), "dice_std": float(d_std),
                         "hd95_mean": float(h_mean), "hd95_std": float(h_std),
                         "hd95_nan_cases": n_nan}
        print(f"  {cls} Dice={d_mean:.4f}+-{d_std:.4f} HD95={h_mean:.2f}+-{h_std:.2f}mm "
              f"(excl={n_nan})", flush=True)
    with open(os.path.join(cfg.output_dir, f"metrics_{split_name}.json"), "w") as f:
        json.dump(results, f, indent=2)
    return results, dice_mat, hd95_mat

In [39]:
# %% Cell 8: Visualisation
 
REGION_COLORS = {
    "WT": np.array([0.0, 1.0, 0.0, 0.35]),
    "TC": np.array([1.0, 1.0, 0.0, 0.40]),
    "ET": np.array([1.0, 0.1, 0.1, 0.55]),
}
 
 
def _make_overlay(mri_sl, et_sl, tc_sl, wt_sl):
    rng  = (mri_sl.max() - mri_sl.min()) + 1e-8
    rgba = np.stack([(mri_sl-mri_sl.min())/rng]*3 + [np.ones_like(mri_sl)], axis=-1)
    for mask, col in [(wt_sl, REGION_COLORS["WT"]),
                      (tc_sl, REGION_COLORS["TC"]),
                      (et_sl, REGION_COLORS["ET"])]:
        m = mask > 0.5; a = col[3]
        rgba[m, :3] = (1-a)*rgba[m, :3] + a*col[:3]
    return rgba
 
 
@torch.no_grad()
def visualise_case_overlays(model, test_cases, cfg, n_cases=4):
    model.eval()
    rng      = np.random.default_rng(SEED)
    selected = rng.choice(len(test_cases), min(n_cases, len(test_cases)), replace=False)
    fig, axes = plt.subplots(len(selected), 3, figsize=(13, 4.5*len(selected)))
    if len(selected) == 1: axes = axes[np.newaxis, :]
    patches = [mpatches.Patch(color=REGION_COLORS[c][:3], label=c) for c in ["WT","TC","ET"]]
    for row, cidx in enumerate(selected):
        case  = test_cases[int(cidx)]
        vols  = [zscore_normalize(load_nifti(case["modalities"][m])) for m in MODALITIES]
        img_np = np.stack(vols, axis=0).astype(np.float32)
        gt_np  = labels_to_channels(load_nifti(case["seg"]))
        logits = sliding_window_inference(
            torch.from_numpy(img_np).unsqueeze(0).to(DEVICE),
            cfg.patch_size, cfg.sw_batch_size, model, overlap=cfg.sw_overlap, mode="gaussian")
        pred_np = (torch.sigmoid(logits[0]) > 0.5).cpu().numpy().astype(float)
        ax_sl   = int(gt_np[2].sum(axis=(1,2)).argmax()); t1c = img_np[1, ax_sl]
        gt_ov   = _make_overlay(t1c, gt_np[0,ax_sl], gt_np[1,ax_sl], gt_np[2,ax_sl])
        pr_ov   = _make_overlay(t1c, pred_np[0,ax_sl], pred_np[1,ax_sl], pred_np[2,ax_sl])
        diff    = ((gt_np[:,ax_sl]>0.5)^(pred_np[:,ax_sl]>0.5)).any(0).astype(float)
        for col, (img, title) in enumerate([(gt_ov,f"GT z={ax_sl}"),(pr_ov,"Pred"),(None,"Error")]):
            ax = axes[row, col]; ax.axis("off"); ax.set_title(title, fontsize=9)
            if img is not None: ax.imshow(img, origin="lower")
            else:
                bg = (t1c-t1c.min())/((t1c.max()-t1c.min())+1e-8)
                ax.imshow(bg, cmap="gray", origin="lower")
                ax.imshow(diff, alpha=0.6, cmap="Reds", vmin=0, vmax=1, origin="lower")
        axes[row,0].set_ylabel(case["case_id"], fontsize=7, rotation=0, labelpad=60, va="center")
    fig.legend(handles=patches, loc="upper right", fontsize=9)
    plt.suptitle("Mod A: Mamba Encoder + CNN Decoder", fontsize=12)
    plt.tight_layout()
    path = os.path.join(cfg.output_dir, "overlay_predictions.png")
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    print(f"  Saved: {path}", flush=True)
 
 
def plot_training_curves(history, cfg):
    ep = list(range(1, len(history["train_loss"])+1))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(ep, history["train_loss"], lw=2, label="Train loss")
    ax1.axvline(cfg.phase1_epochs, color="gray", ls="--", alpha=0.5, label="Ph1->2")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Training Loss")
    ax1.grid(alpha=0.3); ax1.legend()
    for key, lbl, col in [("val_dice_et","ET","red"),("val_dice_tc","TC","gold"),
                            ("val_dice_wt","WT","green"),("val_dice_mean","Mean","steelblue")]:
        ax2.plot(ep, history[key], lw=2, label=lbl, color=col)
    ax2.axvline(cfg.phase1_epochs, color="gray", ls="--", alpha=0.5, label="Ph1->2")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Dice"); ax2.set_title("Val Dice")
    ax2.grid(alpha=0.3); ax2.legend()
    plt.tight_layout()
    path = os.path.join(cfg.output_dir, "training_curves.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"  Saved: {path}", flush=True)
 
 
def plot_metric_summary(dice_mat, hd95_mat, cfg):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    x = np.arange(len(CLASS_NAMES)); colors = ["#E24B4A","#EF9F27","#1D9E75"]
    for ci, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
        d = dice_mat[:,ci]; h = hd95_mat[:,ci]
        j = np.random.default_rng(ci).uniform(-0.1, 0.1, len(d))
        ax1.bar(ci, np.nanmean(d), 0.5, color=col, alpha=0.8, label=cls,
                yerr=np.nanstd(d), capsize=4)
        ax1.scatter(ci+j, d, color="black", s=12, alpha=0.5, zorder=3)
        vh = h[~np.isnan(h)]
        if len(vh):
            j2 = np.random.default_rng(ci+10).uniform(-0.1, 0.1, len(vh))
            ax2.bar(ci, np.nanmean(h), 0.5, color=col, alpha=0.8, label=cls,
                    yerr=np.nanstd(h), capsize=4)
            ax2.scatter(ci+j2, vh, color="black", s=12, alpha=0.5, zorder=3)
    for ax, title, ylabel in [(ax1,"Dice Mod A","Dice"),(ax2,"HD95 Mod A","HD95 (mm)")]:
        ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES); ax.set_title(title)
        ax.set_ylabel(ylabel); ax.legend(); ax.grid(axis="y", alpha=0.3)
    ax1.set_ylim(0, 1.05); plt.tight_layout()
    path = os.path.join(cfg.output_dir, "metric_summary.png")
    plt.savefig(path, dpi=150); plt.close()
    print(f"  Saved: {path}", flush=True)

In [44]:
import gc
import torch

def reset_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()

reset_gpu()

In [46]:
# %% Cell 9: Sanity Checks + Entry Point
 
def sanity_check(model: MambaUNETR3D, cfg: Config):
    print("\n--- Sanity Checks ---", flush=True)
    model.eval()
    B = 1; D = H = W = 64
    x_dummy = torch.zeros(B, cfg.in_channels, D, H, W, device=DEVICE)
    with torch.no_grad():
        x_stem = model.encoder.stem(x_dummy)
        assert x_stem.shape == (B, cfg.feature_size, D//2, H//2, W//2)
        print(f"  [OK] Stem: {tuple(x_dummy.shape)} -> {tuple(x_stem.shape)}", flush=True)
        btk, skips = model.encoder(x_dummy)
        C = cfg.feature_size
        assert btk.shape      == (B, 8*C, D//16, H//16, W//16)
        assert skips[1].shape == (B, C,   D//2,  H//2,  W//2)
        assert skips[2].shape == (B, 2*C, D//4,  H//4,  W//4)
        assert skips[3].shape == (B, 4*C, D//8,  H//8,  W//8)
        print(f"  [OK] Encoder btk={tuple(btk.shape)}", flush=True)
        logits = model(x_dummy)
        assert logits.shape == (B, cfg.num_classes, D, H, W)
        assert not torch.isnan(logits).any()
        print(f"  [OK] Output {tuple(logits.shape)} no NaN", flush=True)
 
    model.train()
    x_tr = torch.randn(B, cfg.in_channels, D, H, W, device=DEVICE)
    y_tr = (torch.rand(B, cfg.num_classes, D, H, W, device=DEVICE) > 0.8).float()
    opt  = torch.optim.AdamW(list(model.decoder.parameters())+list(model.head.parameters()), lr=1e-4)
    scl  = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)
    with torch.amp.autocast("cuda", enabled=cfg.use_amp):
        loss = criterion(model(x_tr), y_tr)
    assert torch.isfinite(loss)
    print(f"  [OK] Loss {loss.item():.4f}", flush=True)
    opt.zero_grad(set_to_none=True); scl.scale(loss).backward(); scl.unscale_(opt)
    dg = [p.grad for p in model.decoder.parameters() if p.grad is not None]
    assert len(dg) > 0 and any(g.abs().max() > 0 for g in dg)
    print(f"  [OK] Gradients: {len(dg)} decoder tensors", flush=True)
    scl.step(opt); scl.update()
 
    model.eval()
    with torch.no_grad():
        pb = (torch.sigmoid(model(x_tr).detach()) > 0.5).long()
        dice_metric(y_pred=pb, y=y_tr.long())
        sc, _ = dice_metric.aggregate(); dice_metric.reset()
    assert sc.shape == (cfg.num_classes,)
    print(f"  [OK] DiceMetric {tuple(sc.shape)} ET={sc[0]:.3f} TC={sc[1]:.3f} WT={sc[2]:.3f}",
          flush=True)
    vram = torch.cuda.max_memory_allocated(DEVICE) / 1e9
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    print(f"  [OK] Peak VRAM {vram:.1f} GB — cache cleared", flush=True)
    print("  All sanity checks PASSED.", flush=True)
 
 
# Entry Point
print("\n" + "="*60, flush=True)
print("  Mod A: Mamba Encoder + CNN Decoder", flush=True)
print("="*60, flush=True)
 
if not Path(cfg.data_root).exists():
    print(f"  Dataset not found — CPU arch check only", flush=True)
    _cfg = Config(); _cfg.feature_size=16; _cfg.depths=(1,1,1,1)
    _cfg.d_state=4; _cfg.d_conv=4; _cfg.mamba_expand=2
    os.makedirs(_cfg.checkpoint_dir, exist_ok=True)
    os.makedirs(_cfg.output_dir, exist_ok=True)
    _m = MambaUNETR3D(_cfg)
    with torch.no_grad(): _y = _m(torch.randn(1,4,64,64,64))
    assert _y.shape == (1,3,64,64,64)
    print(f"  OK (1,4,64^3) -> {tuple(_y.shape)}", flush=True)
    del _m, _y
else:
    (train_loader, val_loader,
     val_vol_loader, test_loader,
     val_cases, test_cases) = build_loaders(cfg)
 
    model = build_model(cfg)
    sanity_check(model, cfg)
 
    # FIX-CKPT: auto-load Phase 1 checkpoint to skip re-running 20 epochs
    _p1_best = os.path.join(cfg.checkpoint_dir, "modA_best.pth")
    _p1_last = os.path.join(cfg.checkpoint_dir, "modA_epoch_last.pth")
    _p1_ckpt = (_p1_best if os.path.exists(_p1_best) else
                _p1_last if os.path.exists(_p1_last) else None)
    if _p1_ckpt is not None:
        _ckpt = torch.load(_p1_ckpt, map_location=DEVICE, weights_only=False)
        if _ckpt.get("phase", 0) >= 1:
            model.load_state_dict(_ckpt["model_state_dict"])
            cfg.phase1_epochs = 0
            print(f"\n  Phase 1 loaded from {os.path.basename(_p1_ckpt)} "
                  f"(ep {_ckpt['epoch']}) — skipping to Phase 2.", flush=True)
 
    model, history = run_training(model, train_loader, val_loader, cfg)
 
    best_path = os.path.join(cfg.checkpoint_dir, "modA_best.pth")
    last_path = os.path.join(cfg.checkpoint_dir, "modA_epoch_last.pth")
    load_path = best_path if os.path.exists(best_path) else last_path
    ckpt_data = torch.load(load_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data["model_state_dict"])
    tag = "best" if "best" in load_path else "epoch_last"
    print(f"\nLoaded {tag} checkpoint (ep {ckpt_data['epoch']})", flush=True)
 
    print("\n--- Validation ---", flush=True)
    val_results, val_dm, val_hm = evaluate_full_volumes(model, val_vol_loader, cfg, "val")
    print("\n--- Test ---", flush=True)
    tst_results, tst_dm, tst_hm = evaluate_full_volumes(model, test_loader, cfg, "test")
 
    print("\n--- Visualisations ---", flush=True)
    plot_training_curves(history, cfg)
    plot_metric_summary(tst_dm, tst_hm, cfg)
    visualise_case_overlays(model, test_cases, cfg, n_cases=4)
 
    print(f"\n  Mod A complete. Ckpts -> {cfg.checkpoint_dir}", flush=True)